# EM27/SUN retrieval demo

Before beginning, please follow the installation instructions in the README of the code repository. Julia needs to be installed, and the required packages must be available on the computer that this notebook runs on. Further, the example data must be present (spectroscopy, solar model, pre-processed EM27 measurements with prior atmosphere).

In this demonstration notebook, we will use pre-processed PROFFAST data (spectra, model atmosphere data) to perform some retrieval using the spectral window **of your choosing**.

This is the slightly simplified version of the other demo notebook. Here another layer is added that hides away most verbose calls in favor of simpler one-liners which may be more instructive.

<div class="alert alert-block alert-info">
<b>ADVICE</b> The retrieval results in this notebook, as well as any comparisons with respect to PROFFAST outputs are for illustration purposes only! The retrievals performed here have <b>not</b> undergone any bias correction - thus the comparisons show <b>raw</b> results against <b>bias-corrected</b> PROFFAST outputs. Further, this demo uses generated spectroscopy data that has not been thoroughly vetted/tuned for this application.
</div>

## Initial set up, loading of modules and functions

In [ ]:
# Activate the local project directory. This allows us to have an encapsulated
# area with the needed packages/modules available without running into version
# conflicts with the "general" Julia environment.
using Pkg; Pkg.activate("./");

In [ ]:
# Load various packages/modules that we need to make the code work

using CSV, DataFrames
using Dates
using Glob
using Interpolations
using LinearAlgebra
using Logging, LoggingExtras
using Plots; default(
    fmt = :svg,
    bottom_margin = 10Plots.mm,
    left_margin = 10Plots.mm,
    top_margin = 10Plots.mm,
    right_margin = 10Plots.mm
)
using Plots.PlotMeasures
using Printf
using ProgressMeter

using Unitful
using Statistics 
using YAML

using RetrievalToolbox; const RE = RetrievalToolbox;

The following lines load some user-defined functions. These are mostly helper functions that make certain tasks easier, e.g. load PROFFAST files, calculate the instrument response functions, etc.

In [ ]:
# Load code to deal with PROFFAST pre-processing outputs
include("PROFFAST_tools.jl");

# Load code to deal with retrieval preparation
include("EM27-retrieval.jl");

# Load the code that contains the forward model
include("forward_model.jl");

# Load the code that processes a scene
# (set up the state vector contents, creates a solver, iterations until convergence is reached)
include("process_scene.jl");

# Load the simplified function layer that automates some of the
# tasks in this demo notebook.
include("simple_layer.jl");

## Retrieval preparation

In this EM27/SUN retrieval demo, we are storing the configuration of the potential retrieval windows in a YAML file called `windows.yml`. Feel free to open that file and study its contents. Each spectral window is identified by a number which represents the order in which the window (generally?) appears in the PROFFAST "invers input file". Look at the `inp_fast` folder inside PROFFAST (or PROFFASTpylot) for a file named `invers24*.inp`, which may have those windows listed. Each entry also includes information whether the spectral data is found in a "SN" or "SM" spectral file, which denotes whether the meaningful data was recorded with the first channel (SN) or the second channel (SM). Finally, each entry requires spectroscopy data: there we define which gases we want to include in the retrieval, and which spectroscopy file and VMR unit we want to use.

**NOTE: This configuration is not part of the core RetrievalToolbox functionality. It is mostly a convenient way of organizing your inputs. We could have arranged this in many ways, but YAML seems a useful method of doing so.**

### Set configuration

In [ ]:
# Load the YAML configuration file `windows.yml`
win_yml, all_idx = load_config();

#### Intermission: the SM and SN spectra files, and their purpose

The function `read_PROFFAST_spectrum` below grabs a mesurement file `*.BIN` and then converts the contents into an easy-to-use dictionary. Note that the *SN.BIN files contain spectra from the *MIR* (mid infra-red) band, up until ~5300 cm-1, 
whereas the *SM.BIN files contain spectral from the *NIR* (near infra-red) band, from ~5500 cm-1 up to ~12500 cm-1. Thus, if we choose a spectral window at ~4800 cm-1, like the strong CO2 band, then we naturally want to use the measurement recorded at the MIR detector. For a spectral window like the O2 absorption at ~7800 cm-1, we must use the NIR detector.

Below is a helpful plot that shows the spectral coverage of the two detectors. Note that not all EM27/SUN instruments come shipped with the MIR detector.

In [ ]:
plot_measurements()

### Load measurements from a folder

Below code snippet will read all measurements from the specified folder and convert them into dictonary-type data. The function `read_measurements` also selects the portions of the spectrum that corresponds to the different spectral windows, so we only keep the measurement data that is relevant.

In [ ]:
measurements = read_measurements(
    "./example_data/spectra/",
    win_yml,
    all_idx
);

In [ ]:
#= 
    For some functions later on, it is useful to have a filename at hand. Some of the file contents
    may be the same for every file, so it does not matter which file we read them from.

    Of course, depending on the spectral window choice, this may be a *SN.BIN or a *SM.BIN file.

=#
fname_first = first_filenames(measurements)

### Visualize the measurements for each spectral range

Below is a series of plots that show the measurements corresponding to the available spectral windows that we set in `windows.yml`. For the sake of illustration the code below just shows the first measurement present in the directory that you chose above. Observe that `measurements` is a dictionary which has keys corresponding to the spectral window numbers from `windows.yml`. Within that is another dictionary which contains the contents of the measurement that can be accessed by the filenames.

In [ ]:
plot_more_measurements(measurements)

### Selecion of spectral window to retrieve

<div class="alert alert-block alert-info">
<b>USER INPUT:</b> In the next cell we choose which retrieval window to use. Select one of the numbers from the `windows.yml` file (1-7) and/or consult the plots just above. Feel free to return to this point later on and change the variable. Note that you must re-run <b>all</b> cells from here onward for the changes to take full effect! You can do that quickly by selecting the cell below, then go to menu item <b>Run</b>, <b>Run Selected Cell and All Below</b>
</div>

In [ ]:
idx_want::Int = 3

In [ ]:
# Helpful to have all measurement file names for this window choice:
all_fnames = collect(keys((measurements[idx_want]))) |> sort;

The next step creates a so-called spectral window according to the specifications that are laid out in the `windows.yml` file. We supply a `buffer` keyword here that makes sure that the high-resolution model grid extends far enough beyond our desired window to allow for the ISRF application at the edge points. We choose this buffer to be some value that should be big enough, in this case ~15 wavenumbers. Look at the ISRF (below) to make sure that the buffer is **larger** than half the width of the ISRF waveform.

In [ ]:
swin_list = create_spectral_windows(
    win_yml,
    idx_want,
    buffer=15.0
)

Similarly, based on the listed spectroscopy objects, we create a gas object for each gas. **NOTE** that this step also loads the large spectroscopy objects into memory, hence this step may take a while. If you have chosen `2` as the spectral window, you will see a warning regarding a 5-dimensional cross section array - feel free to ignore that.

In [ ]:
gas_dict = create_gases(win_yml, idx_want)

### Create the Instrument Spectral Response Function (ISRF)

We calculate the ISRF "on-the-fly" with code that was copied from PROFFAST. The only parameter needed to calculate the ISRF is the so-called modulation efficiency, which is stored in the PROFFAST-preprocessed spectrum files. The ISRFs are pre-calculated and stored in a table. Every spectral sample thus has its own ISRF. This is necessary since the ISRF changes spectrally and we must model this wavenumber-dependent ISRF. The most convenient way is to pre-calculate the ISRF for each spectral sample and then let the software pick the right one during the ISRF application.

In [ ]:
# Produce ISRFs for each window (window index -> ISRF table)
isrf_dict = create_isrf(measurements);

#### Plot the ISRF

In [ ]:
plot_isrf(measurements, isrf_dict)

### Create dispersion objects

Dispersion objects map out the relationship between spectral samples and their wavenumbers. It is important for this relationship to be accurate enough that we can match the modeled spectral features to the measurements. Again, like before, we create these objects and move them into a dictionary where can access them later on.

In [ ]:
dispersions = create_dispersions(measurements, swin_list)

### Interpolate the model atmosphere to the time of measurement

With the PROFFAST workflow, we also get the model atmospheres from the GINPUT tools, derived from GEOS model runs. We take those 3-hourly model files and interpolate the contents to the actual time of measurement. The function that reads those MAP files automatically converts the wet gas mole fractions into dry ones.

In [ ]:
 # Interpolate MAP atmospheres to measurement times   
map_per_file = interpolate_map(
    "./example_data/map/",
    measurements
);

#### Plot the contents of the model atmospheres

In [ ]:
plot_model_atmosphere(measurements)

### Get the pressure data and interpolate at the measurement times

Following line takes all the pressure measurements and supplements the `measurements` dictionaries with the values from the pressure sensor that closest match the measurement time.

In [ ]:
interpolate_pressure_files_B33!(measurements, "./example_data/pressure/b33_20250722.csv")

#### Look at surface pressure data

In [ ]:
plot_pressure_data(measurements)

### Create the state vector!

We must know in advance which state vector elements we plan to use in our retrieval problem. This information is needed to allocate containers that are subsequently used to store Jacobians and other information. For this example, a helper function `create_statevector` (created specficially for this EM27/SUN application), produces the "right" state vector for us. The state vector elements may have no meaningful value in them for the time being, as they will be changed at a later stage. For example, the dispersion polynomial coefficients will be derived from the data inside the `measurements` dictionary, and they will be inserted into the state vector elements before the retrieval begins.

In [ ]:
sv_dict = create_state_vector(dispersions, swin_list)

## Scene set-up

RetrievalToolbox users so-called buffers to store intermediate results, as opposed to creating new vectors and arrays at each evaluation step. In order to do that, we must provide two numbers: one that is slightly larger than the number of high-resolution model wavenumbers (`N1`), and one that is slightly larger than the number of detector elements that we use to compare against the models (`N2`). The code below estimates those numbers for a single-band retrieval set-up.

In [ ]:
buf = create_buffer(measurements, swin_list, dispersions, gas_dict, sv_dict)

The buffer object `buf` is a big, pre-allocated container that stores almost all needed information to run retrievals. However, the function call above only creates suitable and often empy containers, that must still be filled with meaningful information. For example, the buffer has an empty location:

In [ ]:
buf.scene.location

We add meaningful data by creating our own `EarthLocation` object using the coordinates from the measurement data, and then add it to the buffer object (buffer -> scene -> location). We also make sure that the system understands that we are doing retrievals for an uplooking observer.

In [ ]:
ingest_location!(buf, measurements)

# Show the new location object with proper values
# (should be in Greenbelt, Maryland)
buf.scene.location

## Run the retrieval and inspect the result

The code cell below now actually performs one single retrieval for a measurement of choice. The top-level flow is the following: we first pick a number, corresponding to the number of the measurement. We then grab the filename from the list we created earlier (`all_fnames`), and also the measurement dictionary that corresponds to that filename. That measurement is now assigned to the variable `meas`. `meas` contains all the needed per-scene information that changes between measurements (solar angle, the spectrum itself etc.).

We first need to extract the dispersion coefficients from the measurement dictionary: since our dispersion relation is expressed as a polynomial, we can express the first coefficient (order 0, `d0`) as the starting wavenumber and the second coefficient (or the spacing between wavenumbers, order 1, `d1`), as that given by the data as well (stored in `delta_wavenumber`). Note that retrievals for high-resolution spectroscopic measurements are *very sensitive* to this dispersion relationship, and a bad first guess can cause the retrieval to fail (it is not able to "lock in" to the correct value due to the non-linearity of that state vector element).

The `process_spectrum` function takes care of the main process, we just have to pass on the correct values as indicated by the in-line comments in the code cell below.

Once done, we print out some diagnostics, such as the $\chi^2$ statistic of the spectral residual (goodness of fit), as well as the pressure-weighted total columns.

In [ ]:
file_idx = 5 # Pick a number between 1 and length(all_fnames)

solver = process_spectrum(file_idx, all_fnames, measurements, buf);

Code below plots the fit over the measurement, which is always a good way to check if the retrival set-up is able to produce a good fit of the observations. Looking at the spectral residuals too can help diagnose problems.

In [ ]:
plot_fit_result(solver, buf)

## Run the retrieval in a step-by-step fashion

The function `process_scene` is very helpful when wanting to run the full retrieval until it either converges or runs out of iterations. If we want to slowly inspect what happens step by step, we have to make slight changes to the set up. First, we do mostly the same as before, but we set the number of maximal iterations to `0`. The function returns a "solver object", which is the central object that the inversion functions that ship with RetrievalToolbox operate on.

In [ ]:
RE.reset!(solver.state_vector)
solver = process_spectrum(file_idx, all_fnames, measurements, buf; max_iter=0);

If we now want to observe the fit progress step by step, we can manually call for the solver object `s` to perform another iteration. An iteration evaluates the forward model **and** the Jacobians, the inversion algebra then adjusts the state vector contents to minimize the cost function (the difference to the measurement). Call the cell below repeatedly to observe how the retrieval (hopefully) moves its model prediction closer to the observation!

In [ ]:
plot_latest_iteration(solver, buf)
RE.next_iteration!(solver);

## Run a batch of retrievals

The cell below will initiate the processing of many retrievals, the runtime should be around ~5-10 min, depending on computer performance. Results (the total column gas abundances) are collected into a dictionary `result_dict`. The cell below then uses those results to produce some figures where the results are compared against those of the PROFFAST algorithm. 

**Reminder:** the results produced here are **not** bias-corrected, so there are residual, systematic differences compared to those from PROFFAST. They are usually constant offsets as well as solar-angle dependent, which manifests in slopes over time as the sun moves in the sky and the total path of light between the top of the atmosphere and the instrument on the ground changes accordingly.

In [ ]:
result_dict = Dict{String, Any}()

result_dict["SZA"] = Float64[]
result_dict["Date"] = DateTime[]
result_dict["JDate"] = Float64[]

for gas_name in (x -> x.gas_name).(gas_dict[idx_want])
    result_dict[gas_name] = Float64[]
end

@showprogress for file_idx in 1:length(all_fnames)
    
    solver = process_spectrum(file_idx, all_fnames, measurements, buf)

    meas = measurements[idx_want][all_fnames[file_idx]]
        
    xgas = RE.calculate_xgas(buf.scene.atmosphere)

    # Add SZA into results
    push!(result_dict["SZA"], buf.scene.solar_zenith)
    # Add datetimes into results
    push!(result_dict["Date"], meas["datetime"])
    push!(result_dict["JDate"], datetime2julian(meas["datetime"]))
    
    # Add XGAS into results
    for (gas_name, gas_value) in xgas
        push!(result_dict[gas_name], gas_value)
    end

end

Plot the time series that we just processed and compare against the results from the reference algorithm (if available)!

In [ ]:
plot_timeseries(result_dict)